# Setting up

1. Create a working directory and clone the repository.

You will only need to do this once. Before cloning, make sure that you have decided upon a working directory.

**Cloning locally:**

If you are working locally, you can clone the repo into a location of your choice, e.g. your 'Documents' directory. The instructions below guide you through how to do this in your terminal.

First, change your working directory to 'Documents' (or the location of your choice):

```bash
cd ~/Documents
```

Next, create a new folder in 'Documents' with the project title e.g. 'THISTLE_repo':

```bash
mkdir ~/THISTLE_repo  
```

Change your working directory to the project folder:

```bash
cd ~/THISTLE_repo
```

And print the path of your working directory to confirm you are in the right place:

```bash
pwd
```

Now, clone the repo into your working directory:

```bash
git clone https://github.com/jessicawitte92/THISTLE_project
```

Check this worked by printing the contents of your working directory:

```bash
ls
```

**Cloning in Colab:**

You can clone the repo directly into this Colab notebook by using the command `!git clone https://github.com/jessicawitte92/THISTLE_project`. However, this will only clone the repo for the duration of this Colab session. To save files and outputs, you will need to first mount your Google Drive. The cells below will guide you through this process.

First, make sure you are connected to the T4 GPU. In the 'Runtime' menu, click on 'Change runtime type' and select 'T4 GPU' under 'Hardware accelerator'. **NOTE:** you will need to be logged in with your Google account to connect your Drive and to use the GPU. Free access is limited; for larger projects, you may need to consider working on an HPC.

# Image Preprocessing and OCR

This notebook takes raw scanned images of historical newspaper articles (`.png`, `.jpg`, `.jpeg` or `.pdf`) and applies a series of image enhancement steps to prepare them for optical character recognition (OCR). It then extracts machine-readable text from each image using [Tesseract](https://github.com/tesseract-ocr/tesseract), an open-source OCR engine.

**Run cells from top to bottom.** Each cell depends on variables created in earlier cells.



## For working in Colab: Connect your Google Drive

If you will be working in Colab to implement the THISTLE pipeline, you will need to mount your Google Drive in order to save and import data across stages.

To do so, run the code block below. You will need to approve the request to connect your Google Drive account.

If you are working locally, you do not need to mount your Google Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# change to your working directory
%cd "/content/drive/My Drive/THISTLE_project"

## For working in Colab: Clone the repo into your project folder

In [ ]:
!git clone https://github.com/jessicawitte92/THISTLE_project.git

In [ ]:
# check that all contents of the repo cloned:
%ls "/content/drive/My Drive/THISTLE_project/THISTLE_project"

## For working in Colab: Set the path to your working directory

The notebooks assume that your Google Drive top-level directory is called 'My Drive' and contains a subdirectory for this project called 'THISTLE_project' that will serve as your working directory.

If you have assigned a different name to your working directory--or if your Google Drive is organised differently--change the paths in the code block below accordingly.  

In [ ]:
import os

gdrive_path = "/content/drive/My Drive" # change 'My Drive' to your Drive's name (if different)
wdir_path = os.path.join(gdrive_path, "/THISTLE_project/THISTLE_project") # change 'THISTLE_project to your folder's name (if different)

## Step 1: Install dependencies

Run this cell first to install all required Python packages.

- **In Google Colab:** packages are installed for your current session and will need reinstalling if you reconnect.
- **In Anaconda (local):** run this once per environment, or install via `requirements.txt` instead (see `code/README.md`).

> **System requirement:** You must also have **Tesseract OCR** installed on your machine before running this notebook. See `code/README.md` for platform-specific installation instructions. If you are using Google Colab, the cell below handles Tesseract installation automatically.

In [ ]:
# install required Python packages
!pip install pillow pdf2image pytesseract matplotlib

# install Tesseract (Google Colab only — skip this if running locally):
!apt-get install -y tesseract-ocr

## Step 2: Import packages

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageEnhance, ImageFilter
from pdf2image import convert_from_path
import pytesseract as pt

## Step 3: Set your image folder

This code block imports the images in the 'imgs' folder in the THISTLE repo. If you are working with another dataset, update the `img_folder` path below to point to the folder containing your image files. The notebook accepts `.png`, `.jpg`, `.jpeg` and `.pdf` inputs.



In [ ]:
img_folder = "/content/drive/My Drive/THISTLE_project/THISTLE_project/data/imgs"

# check the path is correct; if you do not receive a warning, proceed to the next code block
if not os.path.exists(img_folder):
    print(f'Warning: {img_folder} not found. Please update img_folder to your local path.')

## Step 4: Build the list of image files

This cell scans your image folder and collects all supported image file paths into a list. .png, .jpg., .jpeg and .pdf files are supported. **Note for PDFs:** each page will be treated as a separate image in the next step.

In [ ]:
# collect paths to supported image files
img_files = []

for file in sorted(os.listdir(img_folder)):
    if file.endswith(('.png', '.jpg', '.jpeg', '.pdf')):
        file_path = os.path.join(img_folder, file)
        img_files.append(file_path)

print(f'{len(img_files)} image file(s) found in {img_folder}')

## Step 5: Preprocess images and run OCR

This cell applies four preprocessing steps to each image to improve OCR accuracy:

1. **Greyscale conversion** — removes colour to ensure images are in greyscale
2. **Contrast enhancement** — sharpens the distinction between text and background
3. **Gaussian blur** — reduces noise and softens stray marks
4. **Binarisation** — converts the image to pure black and white using a pixel threshold

For each image, a side-by-side comparison of the original and preprocessed version is displayed. The preprocessed image is then passed to Tesseract for OCR and the result is stored in the `ocr_results` variable.


In [ ]:
# parameters — adjust for your dataset
contrast_factor = 3    # contrast enhancement (>1 increases contrast)
blur_radius = 0.8      # Gaussian blur radius for noise reduction
threshold = 142        # binarisation threshold (0–255)
dpi = 500              # resolution for PDF conversion (ignored for image files)

ocr_results = []

for idx, file_path in enumerate(img_files):
    filename = os.path.basename(file_path)

    # handle PDFs: convert each page to a PIL Image
    if file_path.endswith('.pdf'):
        pages = convert_from_path(file_path, dpi=dpi)
    else:
        pages = [Image.open(file_path)]

    for page_idx, orig_img in enumerate(pages):
        label = filename if len(pages) == 1 else f'{filename} (page {page_idx + 1})'

        # 1. greyscale
        grey_img = orig_img.convert('L')

        # 2. contrast enhancement
        contrast_img = ImageEnhance.Contrast(grey_img).enhance(contrast_factor)

        # 3. noise reduction
        blurred_img = contrast_img.filter(ImageFilter.GaussianBlur(radius=blur_radius))

        # 4. binarisation
        binary_img = blurred_img.point(lambda x: 0 if x < threshold else 255, '1')

        # display before and after
        plt.figure(figsize=(12, 6))
        plt.suptitle(f'Image {idx + 1}: {label}', fontsize=12)

        plt.subplot(1, 2, 1)
        plt.imshow(orig_img)
        plt.title('Original')
        plt.axis('off')

        plt.subplot(1, 2, 2)
        plt.imshow(binary_img, cmap='gray')
        plt.title('Preprocessed')
        plt.axis('off')

        plt.show()

        # OCR with Tesseract
        text = pt.image_to_string(binary_img)
        ocr_results.append({'png_img': label, 'tesseract_ocr': text})
        print(f'  OCR complete: {label}')

print(f'\nDone. {len(ocr_results)} image(s) processed.')

## Step 6: Save OCR output

First, this cell will convert the ocr_results variable to a dataframe. Then, the data will be saved as a .csv file.

**Note:** remember to update `output_path` to your preferred save location.


In [ ]:
df = pd.DataFrame(ocr_results)

output_path = '/content/drive/My Drive/THISTLE_project/processed_imgs.csv'

df.to_csv(output_path, index=False)

# after this cell executes successfully, you should be able to see the new file, 'img_preprocessing_output.csv' in your Drive